## 1. Import libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, adjusted_rand_score

pd.set_option("display.max_columns", None)
np.random.seed(42)

## 2. Unsupervised learning basics

Unsupervised learning finds patterns in data without using target labels during training.

Common tasks:
- **Clustering:** Groups similar observations.
- **Dimensionality reduction:** Represents data with fewer features.
- **Anomaly detection:** Identifies observations that differ from common patterns.

We'll use the Iris dataset for clustering and PCA. The species labels will only be used later to evaluate clustering—not to train K-Means.

In [ ]:
iris = load_iris()
X = pd.DataFrame(iris.data, columns=iris.feature_names)
y = iris.target

print("Dataset shape:", X.shape)
display(X.head())
print("Feature names:", list(X.columns))
print("Species:", list(iris.target_names))

## 3. Standardize the features

Standardization gives each feature a mean near 0 and standard deviation near 1. It is important for distance-based methods such as K-Means and often useful before PCA.

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Scaled feature means:", np.round(X_scaled.mean(axis=0), 3))
print("Scaled feature standard deviations:", np.round(X_scaled.std(axis=0), 3))

## 4. K-Means clustering

K-Means assigns observations to a chosen number of clusters. It iteratively updates cluster centers and assigns each point to its nearest center.

The Iris species labels are not used to fit the model.

In [ ]:
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
clusters = kmeans.fit_predict(X_scaled)

print("First 15 cluster assignments:", clusters[:15])
print("Cluster counts:")
print(pd.Series(clusters).value_counts().sort_index())
print("Silhouette score:", round(silhouette_score(X_scaled, clusters), 3))
print("Adjusted Rand index (evaluation only):", round(adjusted_rand_score(y, clusters), 3))

## 5. Elbow Method

The Elbow Method compares K-Means inertia for different values of K. Inertia measures the sum of squared distances from observations to their assigned cluster centers. Look for a bend where adding more clusters yields diminishing improvement.

In [ ]:
inertias = []
k_values = range(1, 11)

for k in k_values:
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    model.fit(X_scaled)
    inertias.append(model.inertia_)

plt.figure(figsize=(8, 5))
plt.plot(list(k_values), inertias, marker="o")
plt.xlabel("Number of clusters (K)")
plt.ylabel("Inertia")
plt.title("Elbow Method")
plt.xticks(list(k_values))
plt.grid(True)
plt.show()

## 6. PCA basics and 2D projection

Principal Component Analysis (PCA) transforms correlated numerical features into new, perpendicular components ordered by the variance they explain.

We'll reduce the four standardized Iris features to two components so the samples can be plotted on a 2D graph.

In [ ]:
pca_2d = PCA(n_components=2)
X_pca = pca_2d.fit_transform(X_scaled)

print("Original shape:", X_scaled.shape)
print("2D projected shape:", X_pca.shape)
print("Explained variance ratio:", np.round(pca_2d.explained_variance_ratio_, 4))
print("Total variance retained:", round(pca_2d.explained_variance_ratio_.sum() * 100, 2), "%")

plt.figure(figsize=(8, 6))
for i, species in enumerate(iris.target_names):
    mask = y == i
    plt.scatter(X_pca[mask, 0], X_pca[mask, 1], label=species, alpha=0.8)
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.title("Iris Dataset — PCA 2D Projection")
plt.legend()
plt.grid(True)
plt.show()

## 7. Explained variance

`explained_variance_ratio_` reports the fraction of variance captured by each component. The cumulative sum shows how much variance is retained when keeping the first N components.

The exact percentages are calculated by the next cell.

In [ ]:
pca_all = PCA()
pca_all.fit(X_scaled)

variance_ratio = pca_all.explained_variance_ratio_
cumulative_variance = np.cumsum(variance_ratio)

variance_table = pd.DataFrame({
    "Component": [f"PC{i}" for i in range(1, len(variance_ratio) + 1)],
    "Explained variance (%)": variance_ratio * 100,
    "Cumulative variance (%)": cumulative_variance * 100
})
display(variance_table.round(2))

plt.figure(figsize=(8, 5))
plt.bar(range(1, len(variance_ratio) + 1), variance_ratio * 100, label="Individual")
plt.plot(range(1, len(cumulative_variance) + 1), cumulative_variance * 100,
         marker="o", label="Cumulative")
plt.xlabel("Principal component")
plt.ylabel("Explained variance (%)")
plt.title("PCA Explained Variance")
plt.xticks(range(1, len(variance_ratio) + 1))
plt.grid(axis="y")
plt.legend()
plt.show()

## 8. K-Means clustering with PCA

Compare clustering in the original standardized feature space with clustering in the two-component PCA space. Silhouette scores are calculated in each method's own space, so they are useful but not a perfect apples-to-apples comparison. ARI compares assignments with known species labels and is used for evaluation only.

Cluster IDs are arbitrary; cluster 0 does not inherently mean a particular species.

In [ ]:
kmeans_original = KMeans(n_clusters=3, random_state=42, n_init=10)
clusters_original = kmeans_original.fit_predict(X_scaled)

kmeans_pca = KMeans(n_clusters=3, random_state=42, n_init=10)
clusters_pca = kmeans_pca.fit_predict(X_pca)

comparison = pd.DataFrame({
    "Method": ["K-Means on original standardized features", "K-Means on 2D PCA features"],
    "Silhouette score": [
        silhouette_score(X_scaled, clusters_original),
        silhouette_score(X_pca, clusters_pca)
    ],
    "Adjusted Rand index": [
        adjusted_rand_score(y, clusters_original),
        adjusted_rand_score(y, clusters_pca)
    ]
})
display(comparison.round(3))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].scatter(X_pca[:, 0], X_pca[:, 1], c=clusters_original, cmap="viridis")
axes[0].set_title("Clusters from Original Features")
axes[1].scatter(X_pca[:, 0], X_pca[:, 1], c=clusters_pca, cmap="viridis")
axes[1].set_title("Clusters from PCA Features")

for ax in axes:
    ax.set_xlabel("PC1")
    ax.set_ylabel("PC2")
    ax.grid(True)

plt.tight_layout()
plt.show()

## 9. Anomaly detection with Z-score

A Z-score measures how many standard deviations an observation is from the mean. A common simple rule flags absolute Z-scores greater than 3. This rule can miss outliers when extreme values inflate the mean and standard deviation.

In [ ]:
transactions = pd.DataFrame({
    "TransactionAmount": [45, 50, 48, 52, 47, 55, 49, 51, 46, 53, 50, 48, 500, 42, 1000]
})

mean_amount = transactions["TransactionAmount"].mean()
std_amount = transactions["TransactionAmount"].std(ddof=0)

transactions["Z_Score"] = (
    transactions["TransactionAmount"] - mean_amount
) / std_amount
transactions["Z_Anomaly"] = transactions["Z_Score"].abs() > 3

display(transactions)
print("Z-score anomalies:")
display(transactions[transactions["Z_Anomaly"]])

## 10. Anomaly detection with IQR

The Interquartile Range is Q3 − Q1. The common 1.5 × IQR rule flags values below the lower bound or above the upper bound. IQR is less affected by extreme values than the mean and standard deviation.

In [ ]:
q1 = transactions["TransactionAmount"].quantile(0.25)
q3 = transactions["TransactionAmount"].quantile(0.75)
iqr = q3 - q1

lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

transactions["IQR_Anomaly"] = (
    (transactions["TransactionAmount"] < lower_bound) |
    (transactions["TransactionAmount"] > upper_bound)
)

print("Q1:", q1, "Q3:", q3, "IQR:", iqr)
print("Lower bound:", lower_bound, "Upper bound:", upper_bound)
display(transactions)
print("IQR anomalies:")
display(transactions[transactions["IQR_Anomaly"]])

## 11. Distance-based outlier detection

For a small 2D dataset, calculate each point's Euclidean distance from a reference center and flag points beyond a chosen threshold. This is a simple demonstration; the mean center can itself be pulled toward outliers, and the threshold must be chosen for the problem.

In [ ]:
points = np.array([
    [2, 3], [3, 4], [4, 3], [3, 2], [4, 4], [2, 4], [15, 15]
])

center = points.mean(axis=0)
distances = np.sqrt(((points - center) ** 2).sum(axis=1))
threshold = 5
distance_outliers = distances > threshold

distance_results = pd.DataFrame({
    "Feature 1": points[:, 0],
    "Feature 2": points[:, 1],
    "Distance from mean center": distances,
    "Outlier": distance_outliers
})
display(distance_results.round(2))

plt.figure(figsize=(7, 5))
plt.scatter(points[~distance_outliers, 0], points[~distance_outliers, 1], label="Other points")
plt.scatter(points[distance_outliers, 0], points[distance_outliers, 1], label="Flagged points")
plt.scatter(center[0], center[1], marker="x", s=100, label="Mean center")
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.title("Distance-Based Outlier Detection")
plt.legend()
plt.grid(True)
plt.show()

## 12. Real data cleaning: remove or cap outliers

We'll use the IQR bounds calculated above and compare the original transaction values with two treatments:
- **Remove:** Exclude rows outside the IQR bounds.
- **Cap:** Clip values to the lower and upper IQR boundaries.

Only remove records when there is a defensible reason; a large transaction can be valid. Capping changes the recorded value and should be documented.

In [ ]:
original = transactions[["TransactionAmount"]].copy()

is_outlier = (
    (original["TransactionAmount"] < lower_bound) |
    (original["TransactionAmount"] > upper_bound)
)

removed = original.loc[~is_outlier].copy()
capped = original.copy()
capped["TransactionAmount"] = capped["TransactionAmount"].clip(
    lower=lower_bound, upper=upper_bound
)

summary = pd.DataFrame({
    "Method": ["Original", "Outliers removed", "Outliers capped"],
    "Row count": [len(original), len(removed), len(capped)],
    "Mean": [
        original["TransactionAmount"].mean(),
        removed["TransactionAmount"].mean(),
        capped["TransactionAmount"].mean()
    ],
    "Median": [
        original["TransactionAmount"].median(),
        removed["TransactionAmount"].median(),
        capped["TransactionAmount"].median()
    ],
    "Maximum": [
        original["TransactionAmount"].max(),
        removed["TransactionAmount"].max(),
        capped["TransactionAmount"].max()
    ]
})
display(summary.round(2))

plt.figure(figsize=(8, 5))
plt.boxplot(
    [
        original["TransactionAmount"],
        removed["TransactionAmount"],
        capped["TransactionAmount"]
    ],
    labels=["Original", "Removed", "Capped"]
)
plt.ylabel("Transaction amount")
plt.title("Outlier Handling Comparison")
plt.grid(axis="y")
plt.show()